# Hybrid NLP Riz Score Pipeline
## Combining Rule-Based + Pre-trained ML Models

This notebook implements a hybrid approach:
1. Keyword filtering (original method)
2. Named Entity Recognition for Muslim character detection
3. Contextual sentiment analysis around Muslim mentions
4. Zero-shot classification for 5 misrepresentation dimensions
5. Weighted Riz score calculation

In [31]:
# Install required packages (run once)
# !pip install transformers torch sentencepiece protobuf

In [32]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline
import spacy
import en_core_web_sm
from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
# Load data
df = pd.read_csv('final_structured_dialogues.csv')
movie_characters_df = pd.read_csv('movie_characters.csv')
muslim_names_df = pd.read_csv('muslim_names.csv')

print(f"Loaded {len(df)} dialogue lines")
print(f"Loaded {len(movie_characters_df)} character entries")
print(f"Loaded {len(muslim_names_df)} Muslim names")

Loaded 123429 dialogue lines
Loaded 1000 character entries
Loaded 4730 Muslim names


## Step 1: Preprocessing (Same as Original)

In [34]:
# Clean and deduplicate
df = df.fillna('-')
df = df.groupby('Movie Title', group_keys=False).apply(
    lambda x: x.drop_duplicates(subset='Dialogue', keep='first')
)

# Group by movie
df_grouped = df.groupby(["Movie Title", "Year"])["Dialogue"].apply(" ".join).reset_index()
df_grouped.rename(columns={"Dialogue": "Full Script"}, inplace=True)

print(f"Processed {len(df_grouped)} movies")

Processed 60 movies


In [35]:
# Preprocessing function
nlp = en_core_web_sm.load()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(tokens)

df_grouped["Processed Script"] = df_grouped["Full Script"].apply(preprocess_text)
print("Text preprocessing complete")

Text preprocessing complete


## Step 2: Load Pre-trained Models

In [36]:
# Load sentiment analyzer (for contextual analysis)
print("Loading sentiment analysis model...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1  # Use CPU, change to 0 for GPU
)

print("Loading zero-shot classification model...")
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

print("Models loaded successfully!")

Loading sentiment analysis model...


Device set to use cpu


Loading zero-shot classification model...


Device set to use cpu


Models loaded successfully!


## Step 3: Build Muslim Character Dictionary

In [37]:
# Build Muslim name set
muslim_name_set = set(muslim_names_df['Name'].str.lower())

# Extract Muslim characters per movie
movie_muslim_characters = {}

for movie, group in movie_characters_df.groupby('Movie Title'):
    muslim_chars = []
    for character in group['Character'].dropna():
        name_parts = [part.strip(" '\"") for part in str(character).split()]
        if any(part.lower() in muslim_name_set for part in name_parts):
            muslim_chars.append(character)
    movie_muslim_characters[movie] = muslim_chars

# Add to dataframe
df_grouped['Muslim_Characters'] = df_grouped['Movie Title'].map(movie_muslim_characters)
df_grouped['Has_Muslim_Character'] = df_grouped['Muslim_Characters'].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

print(f"Found Muslim characters in {df_grouped['Has_Muslim_Character'].sum()} movies")

Found Muslim characters in 43 movies


## Step 4: Define Misrepresentation Categories

In [38]:
misrepresentation_keywords = {
    "terrorism": [
        "jihad", "jihadist", "suicide bomber", "islamic terrorism", "terror cell", 
        "radical islam", "terror attack", "militant group", "holy war",
        "bomb", "martyr", "radical", "extremist", "militant", "allahu akbar", 
        "god is great", "inshallah", "takbir", "mujahid", "alhamdulillah", 
        "la ilaha illallah", "deen", "umma", "shahada"
    ],
    "anger": [
        "furiously shouted", "slapped", "punched", "violent temper", 
        "uncontrollable rage", "abusive language", "violent outburst", 
        "temper tantrum", "rage", "furious", "short temper", "yelled", 
        "screamed", "snapped", "exploded"
    ],
    "superstition": [
        "divine punishment", "black magic", "sharia", "holy blessing", 
        "superstitious belief", "demonic possession", "primitive tradition", 
        "backward thinking", "religious law", "old ways", "anti-modern", 
        "ancient customs", "superstition", "orthodox"
    ],
    "threat_to_western": [
        "anti-democracy", "death to the west", "sharia law", "dictatorship",
        "destroy freedom", "corrupt government", "overthrow system",
        "gang activity", "underground network", "illegal trade", "contraband", 
        "arms dealer", "trafficking", "fundamentalism"
    ],
    "misogyny": [
        "honor killing", "forced marriage", "child bride", "male authority", 
        "domestic abuse", "rape", "molestation", "harassment", "submission",
        "serve her husband", "woman's duty", "women belong at home", 
        "female obedience", "obedient wife", "women are weak", "man's world", 
        "female servant", "treats women like property", "prostitution", 
        "used for pleasure"
    ]
}

# Define zero-shot labels (more natural language descriptions)
zero_shot_labels = {
    "terrorism": "terrorism, violence, extremism, religious radicalization",
    "anger": "anger, aggression, violent behavior, rage",
    "superstition": "superstition, religious backwardness, primitive beliefs",
    "threat_to_western": "threat to society, anti-democratic values, fundamentalism",
    "misogyny": "misogyny, gender oppression, discrimination against women"
}

## Step 5: Keyword-Based Scoring (Original Method)

In [39]:
# TF-IDF keyword matching (original approach)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_grouped["Processed Script"])
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Initialize keyword scores
for category in misrepresentation_keywords:
    df_grouped[f'{category}_keyword_score'] = 0.0

# Calculate keyword scores
for category, keywords in misrepresentation_keywords.items():
    existing_keywords = [word for word in keywords if word in tfidf_df.columns]
    
    if existing_keywords:
        category_tfidf = tfidf_df[existing_keywords]
        df_grouped[f'{category}_keyword_score'] = category_tfidf.sum(axis=1).values

print("Keyword-based scoring complete")

Keyword-based scoring complete


## Step 6: Contextual Sentiment Analysis Around Muslim Mentions

In [40]:
def extract_muslim_context(script, muslim_chars, window=150):
    """
    Extract sentences/context around Muslim character mentions
    window: number of characters before and after mention
    """
    if not muslim_chars:
        return []
    
    contexts = []
    script_lower = script.lower()
    
    # Look for character name mentions
    for char in muslim_chars:
        # Get first name
        first_name = str(char).split()[0].lower()
        
        # Find all occurrences
        start = 0
        while True:
            pos = script_lower.find(first_name, start)
            if pos == -1:
                break
            
            # Extract context window
            context_start = max(0, pos - window)
            context_end = min(len(script), pos + len(first_name) + window)
            context = script[context_start:context_end]
            
            # Clean and add
            context = context.strip()
            if len(context) > 20:  # Only meaningful contexts
                contexts.append(context)
            
            start = pos + 1
    
    return contexts[:20]  # Limit to 20 contexts per movie

def analyze_sentiment_contexts(contexts):
    """
    Analyze sentiment of contexts using pre-trained model
    Returns negative sentiment ratio
    """
    if not contexts:
        return 0.0
    
    try:
        # Truncate contexts to avoid token limits
        truncated_contexts = [ctx[:512] for ctx in contexts]
        sentiments = sentiment_analyzer(truncated_contexts)
        
        if not isinstance(sentiments, list):
            return 0.0
        
        negative_count = sum(1 for s in sentiments if s['label'] == 'NEGATIVE')
        return negative_count / len(sentiments)
    except:
        return 0.0

print("Analyzing contextual sentiment around Muslim character mentions...")
print("This may take several minutes...")

df_grouped['muslim_contexts'] = df_grouped.apply(
    lambda row: extract_muslim_context(
        row['Full Script'], 
        row['Muslim_Characters']
    ), axis=1
)

df_grouped['sentiment_negative_ratio'] = df_grouped['muslim_contexts'].apply(
    lambda x: analyze_sentiment_contexts(x if isinstance(x, list) and x is not None else [])
)

print("Contextual sentiment analysis complete")

Analyzing contextual sentiment around Muslim character mentions...
This may take several minutes...


TypeError: 'float' object is not iterable

## Step 7: Zero-Shot Classification for 5 Dimensions

In [ ]:
def chunk_script(script, chunk_size=1000, num_chunks=5):
    """
    Split script into manageable chunks for zero-shot classification
    Takes samples from beginning, middle, and end
    """
    script_len = len(script)
    if script_len < chunk_size:
        return [script]
    
    chunks = []
    step = script_len // num_chunks
    
    for i in range(num_chunks):
        start = i * step
        end = min(start + chunk_size, script_len)
        chunks.append(script[start:end])
    
    return chunks

def zero_shot_classify_script(script, labels_dict):
    """
    Classify script chunks using zero-shot classification
    Returns average probabilities for each category
    """
    chunks = chunk_script(script)
    category_scores = {cat: [] for cat in labels_dict.keys()}
    
    for chunk in chunks:
        if len(chunk) < 50:  # Skip very short chunks
            continue
            
        try:
            # Classify against all labels
            all_labels = list(labels_dict.values())
            result = zero_shot_classifier(
                chunk[:1024],  # Limit chunk size
                candidate_labels=all_labels,
                multi_label=True
            )
            
            # Map results back to categories
            label_to_category = {v: k for k, v in labels_dict.items()}
            for label, score in zip(result['labels'], result['scores']):
                category = label_to_category.get(label)
                if category:
                    category_scores[category].append(score)
        except:
            continue
    
    # Average scores across chunks
    avg_scores = {}
    for cat, scores in category_scores.items():
        avg_scores[cat] = np.mean(scores) if scores else 0.0
    
    return avg_scores

print("Running zero-shot classification on scripts...")
print("This will take several minutes per movie...")

zero_shot_results = []
for idx, row in df_grouped.iterrows():
    print(f"Processing {idx+1}/{len(df_grouped)}: {row['Movie Title']}")
    scores = zero_shot_classify_script(row['Full Script'], zero_shot_labels)
    zero_shot_results.append(scores)

# Add zero-shot scores to dataframe
for category in zero_shot_labels.keys():
    df_grouped[f'{category}_zeroshot_score'] = [
        result[category] for result in zero_shot_results
    ]

print("Zero-shot classification complete")

## Step 8: Calculate Weighted Riz Score

In [ ]:
# Define weights for different signals
KEYWORD_WEIGHT = 0.3
ZEROSHOT_WEIGHT = 0.5
SENTIMENT_WEIGHT = 0.2

# Normalize keyword scores (0-1 scale)
for category in misrepresentation_keywords.keys():
    col = f'{category}_keyword_score'
    max_val = df_grouped[col].max()
    if max_val > 0:
        df_grouped[f'{category}_keyword_norm'] = df_grouped[col] / max_val
    else:
        df_grouped[f'{category}_keyword_norm'] = 0.0

# Calculate combined scores for each dimension
for category in misrepresentation_keywords.keys():
    # Combine signals
    keyword_norm = df_grouped[f'{category}_keyword_norm']
    zeroshot = df_grouped[f'{category}_zeroshot_score']
    sentiment = df_grouped['sentiment_negative_ratio']
    
    # Weighted combination
    combined = (
        KEYWORD_WEIGHT * keyword_norm +
        ZEROSHOT_WEIGHT * zeroshot +
        SENTIMENT_WEIGHT * sentiment
    )
    
    df_grouped[f'{category}_combined_score'] = combined

# Calculate final weighted Riz score (0-5 scale)
df_grouped['riz_score_weighted'] = sum(
    df_grouped[f'{cat}_combined_score'] for cat in misrepresentation_keywords.keys()
)

# Also calculate binary flags for comparison (threshold at 0.3)
for category in misrepresentation_keywords.keys():
    df_grouped[f'{category}_flag'] = (
        df_grouped[f'{category}_combined_score'] > 0.3
    ).astype(int)

df_grouped['riz_score_binary'] = sum(
    df_grouped[f'{cat}_flag'] for cat in misrepresentation_keywords.keys()
)

print("Weighted Riz scores calculated")

## Step 9: Results and Comparison

In [ ]:
# Load original results for comparison
original_results = pd.read_csv('final_riz_test_results.csv')

# Merge for comparison
comparison = df_grouped[[
    'Movie Title', 'Year', 'riz_score_weighted', 'riz_score_binary',
    'sentiment_negative_ratio'
]].merge(
    original_results[['Movie Title', 'riz_score']],
    on='Movie Title',
    how='left'
)

comparison.rename(columns={'riz_score': 'riz_score_original'}, inplace=True)
comparison = comparison.sort_values('Year')

print("\n=== COMPARISON: Original vs Hybrid Method ===")
print(comparison[[
    'Movie Title', 'Year', 
    'riz_score_original', 'riz_score_binary', 'riz_score_weighted'
]].head(20))

In [ ]:
# Statistical comparison
print("\n=== STATISTICAL SUMMARY ===")
print(f"Original Method - Mean Riz Score: {comparison['riz_score_original'].mean():.3f}")
print(f"Binary Method - Mean Riz Score: {comparison['riz_score_binary'].mean():.3f}")
print(f"Weighted Method - Mean Riz Score: {comparison['riz_score_weighted'].mean():.3f}")

# Correlation with year
print("\n=== CORRELATION WITH YEAR ===")
print(f"Original: r = {comparison['Year'].corr(comparison['riz_score_original']):.3f}")
print(f"Binary: r = {comparison['Year'].corr(comparison['riz_score_binary']):.3f}")
print(f"Weighted: r = {comparison['Year'].corr(comparison['riz_score_weighted']):.3f}")

# Trendline slopes
import numpy as np
slope_original = np.polyfit(comparison['Year'], comparison['riz_score_original'], 1)[0]
slope_binary = np.polyfit(comparison['Year'], comparison['riz_score_binary'], 1)[0]
slope_weighted = np.polyfit(comparison['Year'], comparison['riz_score_weighted'], 1)[0]

print("\n=== TRENDLINE SLOPES ===")
print(f"Original: {slope_original:.4f}")
print(f"Binary: {slope_binary:.4f}")
print(f"Weighted: {slope_weighted:.4f}")

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Original method
axes[0].scatter(comparison['Year'], comparison['riz_score_original'], 
                color='blue', alpha=0.6, edgecolors='k')
axes[0].plot(comparison['Year'], 
             np.poly1d(np.polyfit(comparison['Year'], comparison['riz_score_original'], 1))(comparison['Year']),
             'r-', linewidth=2)
axes[0].set_title('Original Method (Keyword-Only)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Riz Score')
axes[0].grid(True, alpha=0.3)

# Plot 2: Binary hybrid
axes[1].scatter(comparison['Year'], comparison['riz_score_binary'], 
                color='green', alpha=0.6, edgecolors='k')
axes[1].plot(comparison['Year'], 
             np.poly1d(np.polyfit(comparison['Year'], comparison['riz_score_binary'], 1))(comparison['Year']),
             'r-', linewidth=2)
axes[1].set_title('Hybrid Method (Binary Flags)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Riz Score')
axes[1].grid(True, alpha=0.3)

# Plot 3: Weighted hybrid
axes[2].scatter(comparison['Year'], comparison['riz_score_weighted'], 
                color='purple', alpha=0.6, edgecolors='k')
axes[2].plot(comparison['Year'], 
             np.poly1d(np.polyfit(comparison['Year'], comparison['riz_score_weighted'], 1))(comparison['Year']),
             'r-', linewidth=2)
axes[2].set_title('Hybrid Method (Weighted Scores)')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Riz Score')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('riz_analysis_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'riz_analysis_comparison.png'")

## Step 10: Save Results

In [ ]:
# Prepare detailed output
output_df = df_grouped[[
    'Movie Title', 'Year', 'Has_Muslim_Character',
    'sentiment_negative_ratio',
    'terrorism_combined_score', 'terrorism_flag',
    'anger_combined_score', 'anger_flag',
    'superstition_combined_score', 'superstition_flag',
    'threat_to_western_combined_score', 'threat_to_western_flag',
    'misogyny_combined_score', 'misogyny_flag',
    'riz_score_weighted', 'riz_score_binary'
]].copy()

output_df = output_df.sort_values('Year')
output_df.to_csv('final_riz_test_results_hybrid.csv', index=False)

print("\nResults saved to 'final_riz_test_results_hybrid.csv'")
print("\n=== Sample Results ===")
print(output_df.head(10))

In [ ]:
# Summary statistics by dimension
print("\n=== DIMENSION ANALYSIS ===")
for category in misrepresentation_keywords.keys():
    flagged = output_df[f'{category}_flag'].sum()
    avg_score = output_df[f'{category}_combined_score'].mean()
    print(f"{category.capitalize()}: {flagged} films flagged, avg score = {avg_score:.3f}")

In [ ]:
# Analyze trend change pre/post 2014 (Modi era)
pre_2014 = output_df[output_df['Year'] < 2014]
post_2014 = output_df[output_df['Year'] >= 2014]

print("\n=== PRE/POST 2014 ANALYSIS (Modi Era) ===")
print(f"Pre-2014 Mean Weighted Riz Score: {pre_2014['riz_score_weighted'].mean():.3f}")
print(f"Post-2014 Mean Weighted Riz Score: {post_2014['riz_score_weighted'].mean():.3f}")

change = ((post_2014['riz_score_weighted'].mean() - pre_2014['riz_score_weighted'].mean()) 
          / pre_2014['riz_score_weighted'].mean() * 100)
print(f"\nPercentage Change: {change:+.1f}%")